In [1]:
import os
import trimesh
import cloudvolume as cv

In [9]:
# --- config ---
ROOT_IDS = [
720575940868571544, 720575940879237356, 720575940915390365
]
ROOT_PATH = '/Users/wangchu/psi_neuron_meshes/paper1_figure3_glomerulus/axodendritic'

In [10]:
VISUALIZE_FIRST = False

# --- setup ---
os.makedirs(ROOT_PATH, exist_ok=True)

vol = cv.CloudVolume(
    'graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr',
    secrets='REPLACE_WITH_YOUR_CAVE_TOKEN',
    use_https=True,
    progress=False
)

# --- fetch meshes in one batch ---
print(f'Fetching {len(ROOT_IDS)} meshes…')
mesh_dict = vol.mesh.get(ROOT_IDS)  # dict: {root_id: mesh_obj}

# --- save each as PLY ---
first_visualized = False
for rid in ROOT_IDS:
    mesh_raw = mesh_dict.get(rid, None)
    if mesh_raw is None:
        print(f'[warn] No mesh returned for {rid} (missing/empty). Skipping.')
        continue

    # Build trimesh
    try:
        tm = trimesh.Trimesh(vertices=mesh_raw.vertices, faces=mesh_raw.faces, process=False)
        if tm.vertices.size == 0 or tm.faces.size == 0:
            print(f'[warn] Empty geometry for {rid}. Skipping file write.')
            continue
    except Exception as e:
        print(f'[error] Failed to build trimesh for {rid}: {e}')
        continue

    # Optional quick visualize (only first, to avoid many windows)
    if VISUALIZE_FIRST and not first_visualized:
        try:
            import navis
            import plotly.io as pio
            pio.renderers.default = 'browser'
            mnav = navis.MeshNeuron((mesh_raw.vertices, mesh_raw.faces))
            fig = navis.plot3d(mnav, backend='plotly')
            fig.show()
            first_visualized = True
        except Exception as e:
            print(f'[warn] Visualization failed for {rid}: {e}')

    # Export to PLY
    try:
        ply_bytes = trimesh.exchange.ply.export_ply(tm)
        out_path = os.path.join(ROOT_PATH, f'{rid}.ply')
        with open(out_path, 'wb') as f:
            f.write(ply_bytes)
        print(f'[ok] Saved {rid} → {out_path}')
    except Exception as e:
        print(f'[error] Failed to save {rid}: {e}')

Fetching 3 meshes…
[ok] Saved 720575940868571544 → /Users/wangchu/psi_neuron_meshes/paper1_figure3_glomerulus/axodendritic/720575940868571544.ply
[ok] Saved 720575940879237356 → /Users/wangchu/psi_neuron_meshes/paper1_figure3_glomerulus/axodendritic/720575940879237356.ply
[ok] Saved 720575940915390365 → /Users/wangchu/psi_neuron_meshes/paper1_figure3_glomerulus/axodendritic/720575940915390365.ply
